# LMC2 #333, adım 2: skorlar, üye seçimi, post-processing

L4 ya da A100 ile çalıştır, sonra Çalışma zamanı > Tümünü çalıştır. Bağlantı koparsa yine Tümünü çalıştır, yarım kalan skorlar kaldıkları yerden devam eder.

1. Kurulum ve yol kontrolü
2. Resmi tarza benzer skor: üç smp seed, base held-out, üç Week 3 base ve gruplar (madde 2, 3, 5)
3. smp ensemble ile 8x TTA (madde 7)
4. Modalite raporu (madde 8)
5. Post-processing taraması, held-out eğitim çalışmalarında (madde 9)
6. Çalışma bilgileri (ortam ve sürümler), #333 raporu için

## 1. Kurulum ve yol kontrolü

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob
R = '/content/lmc2-task1-w1'
if not os.path.exists(R):
    !git clone -q https://github.com/neuronautsai/lmc2-task1-w1 {R}
!git -C {R} pull -q && git -C {R} log -1 --oneline
!pip -q install segmentation-models-pytorch==0.6.0 xmltodict
%cd {R}
!nvidia-smi --query-gpu=name,memory.total --format=csv

D = '/content/drive/MyDrive/LMC2'
W5 = f'{D}/outputs/w5'
PAIRS = f'{D}/outputs/dup_check/pairs_excluded.csv'
SMP = [f'{W5}/smp_r34_s42_v2/best.pt', f'{W5}/smp_r34_s123/best.pt', f'{W5}/smp_r34_s456/best.pt']
BASE_HO = f'{W5}/base_heldout_s42/best.pt'
BASE3 = [f'{D}/outputs/w3/base_s{s}/best.pt' for s in (42, 123, 456)]
HELDOUT = f'{W5}/smp_r34_s42_v2/heldout_studies.json'
SIZE = 3000
for p in [PAIRS, *SMP, BASE_HO, *BASE3, HELDOUT]:
    print(os.path.exists(p), p)
    assert os.path.exists(p), f'eksik: {p}'
S_SMP, S_BASE3, S_ALL = ' '.join(SMP), ' '.join(BASE3), ' '.join(SMP + [BASE_HO] + BASE3)

## 2. Tekil modeller ve gruplar, TTA yok

Gruplar: smp3 (üç smp seed), base3 (Week 3 base seed'leri), all6 (ikisi birlikte). Organel bazında seçim aynı split'te yapıldığı için iyimserdir, rapor bunu belirtir.

In [ ]:
O2 = f'{W5}/score_all_hann128'
!python score_official.py --checkpoint {S_ALL} --groups smp3=0,1,2 base3=4,5,6 all6=0,1,2,4,5,6 --window hann --overlap 128 --out {O2} --pairs-csv {PAIRS} --resume

## 3. smp ensemble, 8x TTA

In [ ]:
O3 = f'{W5}/score_smp3_hann128_tta'
!python score_official.py --checkpoint {S_SMP} --groups smp3=0,1,2 --window hann --overlap 128 --tta --out {O3} --pairs-csv {PAIRS} --resume

## 4. Modalite raporu

In [ ]:
!python modality_report.py --score-dir {O2}/smp3
!python modality_report.py --score-dir {O2}/base3

## 5. Post-processing taraması

smp modelleri held-out çalışmaları eğitimde görmedi, seçim orada yapılır. Base modeller bu çalışmalarla eğitildiği için bu taramaya katılmaz.

In [ ]:
O5 = f'{W5}/postprocess_smp3_tta_heldout'
!python postprocess.py --checkpoint {S_SMP} --tta --split heldout --heldout-json {HELDOUT} --out {O5} --pairs-csv {PAIRS} --resume

## 6. Çalışma bilgileri

In [ ]:
for run in ['smp_r34_s42_v2', 'smp_r34_s123', 'smp_r34_s456', 'base_heldout_s42']:
    info = json.load(open(f'{W5}/{run}/run_info.json'))
    keep = {k: info.get(k) for k in ('versions', 'best_epoch', 'train_seconds', 'inference_seconds_per_image',
                                      'peak_gpu_memory_gb', 'parameters', 'heldout_groups', 'macro_ssim', 'macro_pcc')}
    print(run, json.dumps(keep, default=str))
!cat {O2}/overview_macro.csv
!cat {O2}/per_organelle_variants.csv
!cat {O3}/overview_macro.csv